In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA ecommerce;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS checkpoints;
CREATE VOLUME IF NOT EXISTS delta_stream_output;

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", StringType(), True),
    StructField("quantity", IntegerType(), True)
])

In [0]:
stream_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema(schema) \
    .load("/Volumes/workspace/ecommerce/ecommerce_data/")

In [0]:
from pyspark.sql.functions import col

stream_transformed = stream_df.withColumn(
    "revenue",
    col("price") * col("quantity")
)

In [0]:
dbutils.fs.rm("/Volumes/workspace/ecommerce/delta_stream_output/", True)
dbutils.fs.rm("/Volumes/workspace/ecommerce/checkpoints/", True)

In [0]:
query = stream_transformed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/workspace/ecommerce/checkpoints/") \
    .trigger(availableNow=True) \
    .start("/Volumes/workspace/ecommerce/delta_stream_output/")

In [0]:
delta_df = spark.read.format("delta") \
    .load("/Volumes/workspace/ecommerce/delta_stream_output/")

display(delta_df)